# Smart MCQ Solver — Inference Notebook
**Roll No**: 23f3002157 | **Notebook**: DL-23f3002157-notebook-t22026

Models: LSTM · GRU · BERT · DeBERTa · RAG Pipeline

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
kagglehub.dataset_download('risshabsrinivas/mcq-lstm-23f3002157 ')

In [ ]:
!pip install transformers sentence-transformers faiss-cpu torch tqdm -q

In [ ]:
import os, sys, json, pickle, string, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import faiss
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, CrossEncoder
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP   = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
IDX2LABEL   = {v: k for k, v in LABEL_MAP.items()}

TRAIN_PATH  = '/kaggle/input/smart-mcq-solver-challenge/train.csv'
TEST_PATH   = '/kaggle/input/smart-mcq-solver-challenge/test.csv'

LSTM_DIR    = '/kaggle/input/mcq-lstm-23f3002157'
GRU_DIR     = '/kaggle/input/mcq-gru-23f3002157'
BERT_DIR    = '/kaggle/input/mcq-bert-23f3002157'
DEBERTA_DIR = '/kaggle/input/mcq-deberta-23f3002157'
RAG_DIR     = '/kaggle/input/mcq-rag-23f3002157'

print(f'Device: {DEVICE}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f'Train: {train_df.shape} | Test: {test_df.shape}')
train_df.head(3)

## 3. EDA and Statistical Analysis

In [ ]:
# Answer distribution
freq = train_df['answer'].value_counts()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

freq.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].axhline(len(train_df)/5, color='red', linestyle='--', label='Uniform')
axes[0].set_title('Answer Distribution')
axes[0].legend()

train_df['prompt_len'] = train_df['prompt'].apply(lambda x: len(str(x).split()))
axes[1].hist(train_df['prompt_len'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Prompt Length Distribution')
axes[1].set_xlabel('Word Count')

for col in OPTION_COLS:
    axes[2].hist(train_df[col].apply(lambda x: len(str(x).split())),
                 bins=30, alpha=0.4, label=col)
axes[2].set_title('Option Length Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()
print(f'Answer counts:\n{freq}')

In [ ]:
# TF-IDF cosine similarity: prompt vs correct option
all_texts  = train_df['prompt'].tolist()
for col in OPTION_COLS:
    all_texts.extend(train_df[col].tolist())

tfidf_eda  = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_eda.fit(all_texts)

correct_sims, wrong_sims = [], []
for _, row in train_df.iterrows():
    pv = tfidf_eda.transform([str(row['prompt'])])
    for col in OPTION_COLS:
        ov  = tfidf_eda.transform([str(row[col])])
        sim = float(sk_cosine(pv, ov)[0][0])
        (correct_sims if col == row['answer'] else wrong_sims).append(sim)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(correct_sims, bins=40, alpha=0.6, color='green', label=f'Correct (mean={np.mean(correct_sims):.3f})')
ax.hist(wrong_sims,   bins=40, alpha=0.4, color='red',   label=f'Incorrect (mean={np.mean(wrong_sims):.3f})')
ax.set_title('TF-IDF Cosine Similarity: Correct vs Incorrect Options')
ax.legend()
plt.show()

# Hypothesis test
from scipy.stats import mannwhitneyu, chisquare
stat, p = mannwhitneyu(correct_sims, wrong_sims, alternative='greater')
print(f'Mann-Whitney U: stat={stat:.2f}, p={p:.6f}')
obs = freq.values
chi2, p2 = chisquare(obs, f_exp=np.full(5, len(train_df)/5))
print(f'Chi-square (uniform): chi2={chi2:.4f}, p={p2:.6f}')